In [ ]:
import statbotics  
import json  
from datetime import datetime  
  
sb = statbotics.Statbotics()  
print("Statbotics API client initialized")
  
def extract_event_epa_data(event_key: str, output_file: str = None):  
    """Extract post-match EPA data for all teams in an event"""  
      
    # Get all matches for the event  
    matches = sb.get_matches(event=event_key, limit=250)  
      
    event_data = {  
        "event": event_key,  
        "extracted_at": datetime.now().isoformat(),  
        "matches": []  
    }  
      
    for match in matches:  
        match_data = {  
            "match_key": match["key"],  
            "time": match["time"],  
            "status": match["status"],  
            "teams": []  
        }  
          
        # Get team matches for this match (6 teams total)  
        red_teams = match["alliances"]["red"]["team_keys"]  
        blue_teams = match["alliances"]["blue"]["team_keys"]  
          
        for team in red_teams + blue_teams:  
            team_match = sb.get_team_match(team=team, match=match["key"])  
              
            team_epa = {  
                "team": team,  
                "alliance": "red" if team in red_teams else "blue",  
                "epa":{  
                    "pre_match_total": team_match["epa"]["total_points"],  # Pre-match total  
                    "post_match_total": team_match["epa"]["post"],         # Post-match total  
                    "epa_change": team_match["epa"]["post"] - team_match["epa"]["total_points"],  
                    "breakdown": team_match["epa"]["breakdown"]  # Note: still pre-match  
                }  
            }  
            match_data["teams"].append(team_epa)  
        
        event_data["matches"].append(match_data)  
      
    if output_file:  
        with open(output_file, 'w') as f:  
            json.dump(event_data, f, indent=2)  
      
    return event_data  
  
# Example usage  
event_data = extract_event_epa_data("2026orore", "Cais_epa_data.json")  
print(f"Extracted EPA data for {len(event_data['matches'])} matches")